# Transactions EDA & Feature Engineering

Load the cleaned Transactions dataset, explore transaction behaviour, and create reusable fraud-analysis features.

## Analysis Questions

1. What does normal transaction behaviour look like?
2. Are there unusually high/low or negative transaction amounts?
3. Which users have unusually high transaction frequency or value?
4. Which merchants have unusually high transaction frequency or value?
5. Are users connected to many different merchants?
6. Are merchants connected to many different users?
7. Are transactions concentrated at particular hours or days?
8. Are missing UTR/MCC values concentrated among particular users or merchants?
9. Which transactions show multiple potentially suspicious signals?
10. Which features should be carried into the combined KYC + Merchant + Chargeback + Graph analysis?

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT/'data').exists(): ROOT = ROOT.parent
INPUT = ROOT/'data'/'processed'/'transactions_cleaned.csv'
OUTPUT = ROOT/'data'/'processed'/'transactions_features.csv'
print(INPUT, INPUT.exists())

In [ ]:
df = pd.read_csv(INPUT)
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
print('Shape:', df.shape)
display(df.head())
display(df.dtypes)

## 1. Data Quality Check

In [ ]:
quality = pd.DataFrame({'missing_count':df.isna().sum(),'missing_pct':(df.isna().mean()*100).round(2),'unique_values':df.nunique(dropna=True)})
display(quality)
print('Exact duplicate rows:', df.duplicated().sum())
print('Duplicate txn_id:', df.txn_id.duplicated().sum())

## 2. Basic EDA

In [ ]:
print('Negative:', (df.amount<0).sum())
print('Zero:', (df.amount==0).sum())
display(df.amount.describe())
display(df.status.value_counts().to_frame('count'))

In [ ]:
fig, ax = plt.subplots(figsize=(10,5)); df.amount.plot(kind='hist',bins=50,ax=ax); ax.set_title('Transaction Amount Distribution'); ax.set_xlabel('Amount'); ax.set_ylabel('Transactions'); plt.show()

hour = df.timestamp.dt.hour.value_counts().sort_index()
fig, ax = plt.subplots(figsize=(10,5)); hour.plot(kind='bar',ax=ax); ax.set_title('Transactions by Hour'); ax.set_xlabel('Hour'); ax.set_ylabel('Transactions'); plt.xticks(rotation=0); plt.show()

## 3. Transaction-Level Features

In [ ]:
features = df.copy()
features['txn_date'] = features.timestamp.dt.date
features['txn_year'] = features.timestamp.dt.year
features['txn_month'] = features.timestamp.dt.month
features['txn_day_of_week'] = features.timestamp.dt.dayofweek
features['txn_day_name'] = features.timestamp.dt.day_name()
features['txn_hour'] = features.timestamp.dt.hour
features['is_weekend'] = features.txn_day_of_week >= 5
features['is_night'] = features.txn_hour.isin([0,1,2,3,4,5])
features['negative_amount_flag'] = features.amount < 0
features['zero_amount_flag'] = features.amount == 0
features['missing_utr_flag'] = features.utr.isna()
features['missing_mcc_flag'] = features.mcc.isna()
features['amount_abs'] = features.amount.abs()
display(features.head())

## 4. Amount Features

Unusual values are flagged, not deleted, because outliers may be useful fraud signals.

In [ ]:
q1,q3 = features.amount.quantile([.25,.75]); iqr=q3-q1
lo,hi=q1-1.5*iqr,q3+1.5*iqr
features['amount_iqr_outlier_flag'] = (features.amount<lo)|(features.amount>hi)
features['amount_percentile'] = features.amount.rank(pct=True)
print('IQR bounds:',lo,hi)
print('Amount outliers:',features.amount_iqr_outlier_flag.sum())
display(features.loc[features.amount_iqr_outlier_flag,['txn_id','amount','status']].head(20))

## 5. User-Level Features

In [ ]:
user_stats = features.groupby('user_id').agg(
 user_transaction_count=('txn_id','count'), user_total_amount=('amount','sum'), user_avg_amount=('amount','mean'),
 user_max_amount=('amount','max'), user_min_amount=('amount','min'), user_unique_merchants=('merchant_id','nunique'),
 user_negative_txn_count=('negative_amount_flag','sum'), user_missing_utr_count=('missing_utr_flag','sum'), user_missing_mcc_count=('missing_mcc_flag','sum')).reset_index()
features=features.merge(user_stats,on='user_id',how='left')
features['user_negative_txn_ratio']=features.user_negative_txn_count/features.user_transaction_count
display(user_stats.sort_values('user_transaction_count',ascending=False).head(20))

## 6. Merchant-Level Features

In [ ]:
merchant_stats = features.groupby('merchant_id').agg(
 merchant_transaction_count=('txn_id','count'), merchant_total_amount=('amount','sum'), merchant_avg_amount=('amount','mean'),
 merchant_max_amount=('amount','max'), merchant_unique_users=('user_id','nunique'), merchant_negative_txn_count=('negative_amount_flag','sum'),
 merchant_missing_utr_count=('missing_utr_flag','sum'), merchant_missing_mcc_count=('missing_mcc_flag','sum')).reset_index()
features=features.merge(merchant_stats,on='merchant_id',how='left')
features['merchant_negative_txn_ratio']=features.merchant_negative_txn_count/features.merchant_transaction_count
display(merchant_stats.sort_values('merchant_transaction_count',ascending=False).head(20))

## 7. Status Features

Keep original status values. Group variants only for analysis.

In [ ]:
success={'SUCCESS','TXN_SUCCESS','S','COMPLETED'}
failed={'FAILED','TXN_FAILED','FAIL','F','DECLINED'}
pending={'PENDING','PROCESSING','INITIATED'}
features['status_group']=np.select([features.status.isin(success),features.status.isin(failed),features.status.isin(pending)],['SUCCESS','FAILED','PENDING'],'OTHER')
features['success_flag']=features.status_group.eq('SUCCESS')
features['failed_flag']=features.status_group.eq('FAILED')
features['pending_flag']=features.status_group.eq('PENDING')
display(pd.crosstab(features.status,features.status_group))

## 8. User–Merchant Relationship Features

In [ ]:
pair_stats=features.groupby(['user_id','merchant_id']).agg(user_merchant_txn_count=('txn_id','count'),user_merchant_total_amount=('amount','sum'),user_merchant_avg_amount=('amount','mean')).reset_index()
features=features.merge(pair_stats,on=['user_id','merchant_id'],how='left')
display(pair_stats.sort_values('user_merchant_txn_count',ascending=False).head(20))

## 9. User Velocity Features

These capture rapid transaction bursts. They are signals for investigation, not proof of fraud.

In [ ]:
features=features.sort_values(['user_id','timestamp']).copy()
features['user_prev_txn_timestamp']=features.groupby('user_id').timestamp.shift(1)
features['minutes_since_prev_user_txn']=(features.timestamp-features.user_prev_txn_timestamp).dt.total_seconds()/60
features['rapid_user_txn_flag']=features.minutes_since_prev_user_txn.notna() & (features.minutes_since_prev_user_txn<=5)
display(features[['user_id','timestamp','minutes_since_prev_user_txn','rapid_user_txn_flag']].head(20))

## 10. Multi-Signal Investigation Flag

This is deliberately not the final fraud score. It counts simple transaction-level signals for exploration.

In [ ]:
signals=['negative_amount_flag','amount_iqr_outlier_flag','missing_utr_flag','missing_mcc_flag','failed_flag','rapid_user_txn_flag','is_night']
features['transaction_signal_count']=features[signals].astype(int).sum(axis=1)
features['multiple_signal_flag']=features.transaction_signal_count>=2
display(features.sort_values(['transaction_signal_count','amount_abs'],ascending=[False,False])[['txn_id','user_id','merchant_id','amount','status','transaction_signal_count']+signals].head(30))

## 11. Top Users and Merchants for Investigation

In [ ]:
top_users=features.groupby('user_id').agg(transactions=('txn_id','count'),total_amount=('amount','sum'),avg_amount=('amount','mean'),unique_merchants=('merchant_id','nunique'),negative_transactions=('negative_amount_flag','sum'),rapid_transactions=('rapid_user_txn_flag','sum'),multiple_signal_transactions=('multiple_signal_flag','sum')).reset_index()
top_merchants=features.groupby('merchant_id').agg(transactions=('txn_id','count'),total_amount=('amount','sum'),avg_amount=('amount','mean'),unique_users=('user_id','nunique'),negative_transactions=('negative_amount_flag','sum'),multiple_signal_transactions=('multiple_signal_flag','sum')).reset_index()
print('Top users'); display(top_users.sort_values('transactions',ascending=False).head(20))
print('Top merchants'); display(top_merchants.sort_values('transactions',ascending=False).head(20))

## 12. Export and Validation

In [ ]:
features=features.sort_values('timestamp').reset_index(drop=True)
print('Original rows:',len(df)); print('Feature rows:',len(features)); print('Unique txn_id:',features.txn_id.nunique()); print('Feature columns:',len(features.columns))
assert len(features)==len(df), 'Feature engineering changed transaction row grain.'
assert features.txn_id.nunique()==len(features), 'txn_id is no longer unique.'
OUTPUT.parent.mkdir(parents=True,exist_ok=True)
features.to_csv(OUTPUT,index=False)
print('Saved:',OUTPUT)

## Key Findings to Record

- Amount behaviour: Transaction amounts range from -24,847.86 to 24,998.12, with a median of approximately 12,218. The dataset contains both positive and negative transaction amounts, so unusual amounts should be flagged rather than removed.
- Negative transaction pattern: There are 420 negative transactions after duplicate removal. Negative amounts are unusual and should be treated as an investigation signal rather than automatically considered fraudulent.
- Status pattern: The majority of transactions are successful. Approximately 85.3% are successful, 9.8% failed, and 5.0% are pending after grouping equivalent status variants for analysis. No single status pattern by itself indicates fraud.
- Time pattern: Transaction activity varies across hours and dates. Time-based features such as transaction hour, day of week, weekend flag and night-time flag are useful for identifying unusual transaction behaviour and rapid activity.
- User behaviour: Users differ in transaction frequency, total transaction value, average transaction amount and number of unique merchants used. These differences can be used to identify users with unusually high activity or unusually broad merchant connections.
- Merchant behaviour: Merchants vary in transaction volume, total transaction value, average transaction amount and number of unique users. Merchant-level activity is therefore useful for identifying unusually active or unusual merchant accounts.
- User–merchant relationship pattern: The same user can transact with multiple merchants, while merchants can receive transactions from multiple users. The user–merchant transaction count, total amount and average amount are useful relationship features for later graph analysis.
- Potential suspicious patterns: Potential investigation signals include negative amounts, unusually large amounts, missing UTR/MCC, failed transactions, night-time activity and rapid consecutive transactions. A transaction showing multiple signals deserves higher investigation priority, but these signals alone do not prove fraud.
- Features to carry into combined analysis: Key features include amount, amount outlier flag, negative amount flag, transaction status group, hour, day of week, weekend/night flag, missing UTR/MCC flags, user transaction count, user total/average amount, unique merchants, merchant transaction count, merchant total/average amount, unique users, user–merchant transaction count, transaction velocity and multi-signal flag.
